# Ben & Jerry's claw machine at UTown: geometric analysis

Question: the machine charges $1 per play and a small tub costs ~$6.
What distribution models the number of tries until a successful grab, and
how do we estimate its parameter from data?

**Answer:** $T \sim \mathrm{Geo}(p)$, $P(T=t)=(1-p)^{t-1}p$, with MLE
$\hat p = \text{grabs} / \text{total plays}$. The game breaks even when
$p = 1/6$.

This notebook walks through the estimation on the (simulated) example data.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

sys.path.insert(0, str(Path.cwd() / "analysis"))

from fit_geometric import fit_geometric, chi2_gof
import analyze

%matplotlib inline
plt.rcParams["figure.dpi"] = 110

In [ ]:
# Load the (simulated) example data and fit the geometric model
plays, grabbed = analyze.load_data(Path("data/example_observations.csv"))
fit = fit_geometric(plays, grabbed, cost_per_play=1.0, tub_price=6.0)

print(f"sessions        : {fit.n_success + fit.n_censored} ({fit.n_success} grabbed, {fit.n_censored} gave up)")
print(f"total plays     : {fit.n_plays}")
print(f"p_hat (MLE)     : {fit.n_success}/{fit.n_plays} = {fit.p:.5f}")
print(f"SE              : {fit.se:.5f}")
print(f"95% Wilson CI   : [{fit.ci_wilson[0]:.5f}, {fit.ci_wilson[1]:.5f}]")
print(f"expected tries  : {fit.expected_tries:.2f}  ->  expected spend ${fit.expected_cost:.2f}")
print(f"break-even p*   : 1/6 = {1/6:.4f}")

In [ ]:
# Model checks: goodness of fit + test against the break-even p*
gof = chi2_gof(plays[grabbed == 1], fit.p)
if gof is not None:
    chi2, dof, pv, ok = gof
    print(f"chi-square vs fitted geometric: chi2 = {chi2:.3f}, df = {dof}, p = {pv:.3f}")

p_star = 1 / 6
print(f"binomial test  H0: p = p* = 1/6  ->  two-sided p = "
      f"{stats.binomtest(fit.n_success, fit.n_plays, p_star).pvalue:.4f}")

In [ ]:
# Visual check 1: observed try counts vs fitted geometric PMF
t = plays[grabbed == 1]
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.hist(t, bins=np.arange(0.5, t.max() + 1.5, 1.0), density=True,
        alpha=0.7, color="#5b8ff9", label=f"observed (n={len(t)}, mean={t.mean():.2f})")
x = np.arange(1, int(t.max()) + 1)
ax.plot(x, (1 - fit.p) ** (x - 1) * fit.p, "o-", color="#e8684a",
        label=f"fitted Geo(p_hat={fit.p:.3f})")
ax.set_xlabel("plays until first successful grab  $t$")
ax.set_ylabel("probability")
ax.legend()
plt.show()

In [ ]:
# Visual check 2: survival curve on a log scale (should be log-linear)
fig, ax = plt.subplots(figsize=(7, 4.2))
xs = np.arange(1, int(t.max()) + 1)
ax.step(xs, [(t > v).mean() for v in xs], where="post", color="#5b8ff9",
        label="empirical $\\hat S(t)$")
ax.plot(xs, (1 - fit.p) ** xs, "--", color="#e8684a", label="geometric $(1-\\hat p)^t$")
ax.set_yscale("log")
ax.set_xlabel("$t$ (plays)")
ax.set_ylabel("$P(T > t)$")
ax.legend()
plt.show()

## Interpretation (example data)

The example data is **simulated** from $\mathrm{Geo}(p = 1/6)$ with a
fixed seed — it is a reproducible illustration, not an observation from the
UTown machine.

* $\hat p = 0.202$ (95% CI $[0.160, 0.251]$): the estimate is consistent
  with the break-even $p^{*} = 1/6$ (binomial test $p = 0.10$); with only
  297 plays the CI is still wide.
* The chi-square test does not reject the geometric model
  ($p = 0.84$), and the log-scale survival curve is close to linear —
  no evidence that the per-play success probability changes with $t$.

To get the *real* answer, collect sessions per
`docs/DATA_COLLECTION.md`, save them as `data/observations.csv`, and
re-run this notebook.